## Projekt: Solver równań płytkiej wody dla morza metanowego Tytana -  fale nad podwodnymi przeszkodami

### 0. potrzebne pakiety Pythona

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import pyplot
from open_atmos_jupyter_utils import show_plot, show_anim
from PyMPDATA import ScalarField, Solver, Stepper, VectorField, Options, boundary_conditions
%config InlineBackend.figure_format = 'pdf'

### 1. opis układu: symbole i równania

$$ \zeta(t; x,y) \rightarrow \text{wysokość swobodnej powierzchni względem geoidy (z=0)}$$
$$ b(x,y) \rightarrow \text{batymetria mierzona dodatnio w doół od geoidy}$$
$$ h(t; x,y) = \zeta + b \rightarrow \text{całkowita głębokość kolumny wody} $$
$$ \vec{u} = [u, v]$$
$$
\begin{cases}
  \partial_t h   &= -\nabla \cdot (\vec{u}h)\\
  \partial_t(hu) &= -\nabla \cdot (\vec{u}hu) - gh\,\partial_x\zeta \\
  \partial_t(hv) &= -\nabla \cdot (\vec{u}hv) - gh\,\partial_y\zeta
\end{cases}
$$

### 2. Solver równań płytkiej wody zbudowany na bazie PyMPDATA

In [4]:
class ShallowWaterEquationsIntegrator:
    def __init__(self, *, h_initial: np.ndarray, options: Options = None):
        """ initializes the solvers for a given initial condition of `h` assuming zero momenta at t=0 """
        options = options or Options(nonoscillatory=True, infinite_gauge=True)
        X, Y, grid = 0, 1, h_initial.shape
        stepper = Stepper(options=options, grid=grid)
        kwargs = {
            'boundary_conditions': [boundary_conditions.Constant(value=0)] * len(grid),
            'halo': options.n_halo,
        }
        advectees = {
            "h": ScalarField(h_initial, **kwargs),
            "uh": ScalarField(np.zeros(grid), **kwargs),
            "vh": ScalarField(np.zeros(grid), **kwargs),
        }
        self.advector = VectorField((
                np.zeros((grid[X] + 1, grid[Y])),
                np.zeros((grid[X], grid[Y] + 1))
            ), **kwargs
        )
        self.solvers = { k: Solver(stepper, v, self.advector) for k, v in advectees.items() }

    def __getitem__(self, key):
        """ returns `key` advectee field of the current solver state """
        return self.solvers[key].advectee.get()
    
    def _apply_half_rhs(self, *, key, axis, g_times_dt_over_dxy):
        """ applies half of the source term in the given direction """
        self[key][:] -= .5 * g_times_dt_over_dxy * self['h'] * np.gradient(self['h']-bathymetry, axis=axis)

    def _update_courant_numbers(self, *, axis, key, mask, dt_over_dxy):
        """ computes the Courant number component from fluid column height and momenta fields """
        velocity = np.where(mask, np.nan, 0)
        momentum = self[key]
        np.divide(momentum, self['h'], where=mask, out=velocity)

        # using slices to ensure views (over copies)
        all = slice(None, None) 
        all_but_last = slice(None, -1)
        all_but_first_and_last = slice(1, -1)

        velocity_at_cell_boundaries = velocity[( 
            (all_but_last, all),
            (all, all_but_last),
        )[axis]] + np.diff(velocity, axis=axis) / 2 
        courant_number = self.advector.get_component(axis)[(
            (all_but_first_and_last, all),
            (all, all_but_first_and_last)
        )[axis]]
        courant_number[:] = velocity_at_cell_boundaries * dt_over_dxy[axis]
        assert np.amax(np.abs(courant_number)) <= 1

    def __call__(self, *, nt: int, g: float, dt_over_dxy: tuple, outfreq: int, eps: float=1e-7):
        """ integrates `nt` timesteps and returns a dictionary of solver states recorded every `outfreq` step[s] """
        output = {k: [] for k in self.solvers.keys()}
        for it in range(nt + 1): 
            if it != 0:
                mask = self['h'] > eps 
                for axis, key in enumerate(("uh", "vh")):
                    self._update_courant_numbers(axis=axis, key=key, mask=mask, dt_over_dxy=dt_over_dxy)
                self.solvers["h"].advance(n_steps=1)
                for axis, key in enumerate(("uh", "vh")):
                    self._apply_half_rhs(key=key, axis=axis, g_times_dt_over_dxy=g * dt_over_dxy[axis])
                    self.solvers[key].advance(n_steps=1)
                    self._apply_half_rhs(key=key, axis=axis, g_times_dt_over_dxy=g * dt_over_dxy[axis])
            if it % outfreq == 0:
                for key in self.solvers.keys():
                    output[key].append(self[key].copy())
        return output

### 3. Konfiguracja układu i parametrów symulacji

In [5]:
# fizyczne parametry Tytana
g_titan = 1.352  # m/s^2
title_suffix = "Pole prędkości"

def make_grid(nx=60, ny=40):
    x = np.arange(nx)
    y = np.arange(ny)
    X, Y = np.meshgrid(x, y, indexing="ij")
    return x, y, X, Y


# BATYMETRIA: zbocze + pojedyncza podwodna góra
def bathymetry_slope_with_bump(nx, ny, X, Y, slope_max=50.0, bump_amplitude=-20.0, sigma_x_fac=1/15, sigma_y_fac=1/10):
    # zbocze: x=0 głęboko, x=nx-1 płytko
    x = np.arange(nx)
    slope = slope_max * (1 - x / (nx - 1))  # [0, slope_max]
    b = np.tile(slope[:, np.newaxis], (1, ny))

    # garb w środku
    x0 = int(nx * 0.5)
    y0 = int(ny * 0.5)

    sigma_x = nx * sigma_x_fac
    sigma_y = ny * sigma_y_fac

    bump = bump_amplitude * np.exp(
        -(((X - x0) ** 2) / (2 * sigma_x ** 2)
          + ((Y - y0) ** 2) / (2 * sigma_y ** 2))
    )

    b = b + bump
    b = np.maximum(b, 1.0)
    return b


# BATYMETRIA: dwie góry (cieśnina)
def bathymetry_two_ridges(nx, ny, X, Y,base_depth=40.0, slope_max=20.0,bump_amp=-35.0, sigma_fac=1/12):
                          
    x = np.arange(nx)
    slope = base_depth + slope_max * (x / (nx - 1))  
    b = np.tile(slope[:, np.newaxis], (1, ny))

    # wspólna szerokość w x i y
    sigma = nx * sigma_fac

    # położenie gór
    x_mount = int(nx * 0.45)
    y1 = int(ny * 0.70)
    y2 = int(ny * 0.30)

    bump1 = bump_amp * np.exp(
        -(((X - x_mount)**2) / (2 * sigma**2) +
          ((Y - y1)      **2) / (2 * sigma**2))
    )
    bump2 = bump_amp * np.exp(
        -(((X - x_mount)**2) / (2 * sigma**2) +
          ((Y - y2)      **2) / (2 * sigma**2))
    )

    b = b + bump1 + bump2
    b = np.maximum(b, 2.0)  # min 2 m wody
    return b


# FALA: płaska, Gauss w x, stała w y
def initial_flat_wave(nx, ny, X, Y, amp=1.5,x_frac=0.15, sigma_fac=1/15):
    x_wave = int(nx * x_frac)
    sigma_wave = nx * sigma_fac
    zeta = amp * np.exp(-((X - x_wave)**2) / (2 * sigma_wave**2))
    return zeta


# FALA: zaburzenie w głębszej części z lekkim „tiltem”
def initial_gaussian_tilted(nx, ny, X, Y, amp=1.0, x_frac=0.2, sigma_x_fac=1/12, sigma_y_fac=1/4,tilt_strength=0.3):
    x0 = int(nx * x_frac)
    y0 = int(ny * 0.5)
    sigma_x = nx * sigma_x_fac
    sigma_y = ny * sigma_y_fac

    zeta = amp * np.exp(
        -(((X - x0) ** 2) / (2 * sigma_x ** 2)
          + ((Y - y0) ** 2) / (2 * sigma_y ** 2))
    )
    tilt = 1.0 + tilt_strength * (x0 - X) / nx
    zeta *= tilt
    zeta = np.clip(zeta, -0.1, 1.5)
    return zeta

In [6]:
def plot(frame, *, zlim=(-1.5, 1.5)):
    # wysokość swobodnej powierzchni względem dna (ζ)
    psi = output['h'][frame] - bathymetry

    xi, yi = np.indices(psi.shape)
    fig, ax = pyplot.subplots(subplot_kw={"projection": "3d"}, figsize=(12, 6))
    
    #  wireframe
    ax.plot_wireframe(xi + 0.5, yi + 0.5, psi, color='blue', linewidth=0.6, rstride=2, cstride=2)

    ax.set(
        zlim=zlim,
        proj_type='ortho',
        title=f"Tytan – ewolucja powierzchni cieczy nad ukształtowaniem dna (klatka {frame})",
        zlabel=r"$\zeta$ [m]"
    )

    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.fill = False
        axis.pane.set_edgecolor('black')
        axis.pane.set_alpha(1)

    for axis in ('x', 'y'):
        getattr(ax, f'set_{axis}label')(f"{axis}")

    # rzut batymetrii na dno powierzchni
    cf = ax.contourf(
        xi + 0.5, yi + 0.5, bathymetry,
        zdir='z', offset=zlim[0],
        cmap='terrain_r', alpha=0.5 
    )
    cbar = pyplot.colorbar(cf, pad=.1, aspect=10, fraction=.02,
                           label=fr'Głębokość dna $b$ [m]', location='left')
    cbar.ax.invert_yaxis() # głębokość rośnie w dół

    # Ustawienie kamery dla lepszego widoku
    ax.view_init(elev=40, azim=-60)
    return fig


def plot_velocity_field(frame):
    h = output['h'][frame]
    uh = output['uh'][frame]
    vh = output['vh'][frame]

    # prędkości u, v
    u = np.zeros_like(uh)
    v = np.zeros_like(vh)
    mask = h > 1e-8
    u[mask] = uh[mask] / h[mask]
    v[mask] = vh[mask] / h[mask]

    speed = np.sqrt(u**2 + v**2)

    nx_loc, ny_loc = h.shape

    xi = np.arange(nx_loc)
    yi = np.arange(ny_loc)
    Xg, Yg = np.meshgrid(xi, yi, indexing='ij')

    fig, ax = plt.subplots(figsize=(10, 5))

    im = ax.imshow(speed.T, origin='lower', cmap='viridis', aspect='equal')
    cbar = fig.colorbar(im, ax=ax, label=r'Moduł prędkości $|\mathbf{u}|$ [j.u.]')

    step = 2
    X_sub = Xg[::step, ::step]
    Y_sub = Yg[::step, ::step]
    u_sub = u[::step, ::step]
    v_sub = v[::step, ::step]

    ax.quiver(X_sub,Y_sub,u_sub,v_sub,color='white',scale=10, width=0.006,)
    ax.set_xlabel("x / Δx")
    ax.set_ylabel("y / Δy")
    ax.set_title(f"{title_suffix} – pole prędkości, klatka {frame}")
    fig.tight_layout()
    return fig

## Definicje funkcji dla trzech przypadków symulacji

### Zbocze z pojedynczą podwodną górą – skośne zaburzenie początkowe

In [7]:
def run_jedna_gora_tilted_wave(nx=60, ny=40, nt=170,dt_over_dxy=(0.05, 0.05),outfreq=2,):
    global bathymetry, output, title_suffix

    nx, ny = nx, ny 

    x, y, X, Y = make_grid(nx, ny)
    title_suffix = "Zbocze z podwodną górą, zaburzenie skośne"

    bathymetry = bathymetry_slope_with_bump(nx, ny, X, Y)
    zeta_initial = initial_gaussian_tilted(nx, ny, X, Y)
    h_initial = bathymetry + zeta_initial

    integrator = ShallowWaterEquationsIntegrator(h_initial=h_initial)
    output = integrator(nt=nt, g=g_titan, dt_over_dxy=dt_over_dxy,outfreq=outfreq) 
   
    return x, y, X, Y, bathymetry, zeta_initial, h_initial, output

### Zbocze z pojedynczą podwodną górą – nadchodząca fala płaska

In [8]:
def run_jedna_gora_flat_wave(nx=60, ny=40,nt=170,dt_over_dxy=(0.05, 0.05),outfreq=2,amp=1.5,x_frac=0.15,):
    global bathymetry, output, title_suffix

    nx, ny = nx, ny

    x, y, X, Y = make_grid(nx, ny)
    title_suffix = "Zbocze z podwodną górą, fala płaska"

    bathymetry = bathymetry_slope_with_bump(nx, ny, X, Y)
    zeta_initial = initial_flat_wave(nx, ny, X, Y, amp=amp, x_frac=x_frac)
    h_initial = bathymetry + zeta_initial

    integrator = ShallowWaterEquationsIntegrator(h_initial=h_initial)
    output = integrator( nt=nt,  g=g_titan,  dt_over_dxy=dt_over_dxy, outfreq=outfreq)

    return x, y, X, Y, bathymetry, zeta_initial, h_initial, output

### Cieśnina z dwiema podwodnymi górami – propagacja fali płaskiej

In [9]:
def run_ciesnina(nx=60, ny=40,nt=170, dt_over_dxy=(0.05, 0.05), outfreq=2,amp=1.5,x_frac=0.15):
    global bathymetry, output, title_suffix
    nx, ny = nx, ny

    x, y, X, Y = make_grid(nx, ny)
    title_suffix = "Cieśnina z dwiema podwodnymi górami, fala płaska"

    bathymetry = bathymetry_two_ridges(nx, ny, X, Y)
    zeta_initial = initial_flat_wave(nx, ny, X, Y, amp=amp, x_frac=x_frac)
    h_initial = bathymetry + zeta_initial

    integrator = ShallowWaterEquationsIntegrator(h_initial=h_initial)
    output = integrator(nt=nt,g=g_titan,dt_over_dxy=dt_over_dxy, outfreq=outfreq)

    return x, y, X, Y, bathymetry, zeta_initial, h_initial, output

## Wyniki dla trzech scenariuszy: uruchomienie i wizualizacja


### Zbocze z podwodną górą, zaburzenie skośne

In [ ]:
x1, y1, X1, Y1, b1, z1, h1, out1 = run_jedna_gora_tilted_wave(nx=60, ny=40,nt=170, dt_over_dxy=(0.05, 0.05),outfreq=2)
plt.figure(figsize=(8, 4))
plt.imshow(h1.T, origin='lower', cmap='terrain_r')
plt.colorbar(label=fr'Całkowita głębokość $h$ [m]')
plt.title('Profil początkowy – zbocze z podwodną górą, zaburzenie skośne')
plt.xlabel('x [km]')
plt.ylabel('y [km]')
plt.tight_layout()
plt.savefig('1_1_Profil_poczatkowy_zbocze_z_podwodna_gora_zaburzenie_skosne.pdf', format='pdf', bbox_inches='tight')
# show_anim(plot, range(len(out1['h'])), gif_file = '1_2_Ewolucja_powierzchni_zbocze_z_podwodna_gora_zaburzenie_skosne.gif')
# show_anim(plot_velocity_field, range(len(out1['h'])), gif_file='1_3_Pole_predkosci_zbocze_z_podwodna_gora_zaburzenie_skosne.gif')

<Figure size 800x400 with 2 Axes>

### Zbocze z podwodną górą, fala płaska

In [ ]:
x2, y2, X2, Y2, b2, z2, h2, out2 = run_jedna_gora_flat_wave(nx=60, ny=40, nt=170,dt_over_dxy=(0.05, 0.05), outfreq=2,
                                                            amp=1.5,x_frac=0.15)
plt.figure(figsize=(8, 4))
plt.imshow(h2.T, origin='lower', cmap='terrain_r')
plt.colorbar(label=fr'Całkowita głębokość $h$ [m]')
plt.title('Profil początkowy – zbocze z podwodną górą, fala płaska')
plt.xlabel('x [km]')
plt.ylabel('y [km]')
plt.tight_layout()
plt.savefig('2_1_Profil_poczatkowy_zbocze_z_podwodna_gora_fala_plaska.pdf', format='pdf', bbox_inches='tight')

# show_anim(plot, range(len(out2['h'])), gif_file='2_2_Ewolucja_powierzchni_zbocze_z_podwodna_gora_fala_plaska.gif')
# show_anim(plot_velocity_field, range(len(out2['h'])), gif_file = '2_3_Pole_predkosci_zbocze_z_podwodna_gora_fala_plaska.gif')

<Figure size 800x400 with 2 Axes>

### Cieśnina z dwiema podwodnymi górami, fala płaska

In [ ]:
x3, y3, X3, Y3, b3, z3, h3, out3 = run_ciesnina(nx=60, ny=40,nt=170, dt_over_dxy=(0.05, 0.05),outfreq=2, amp=1.5,x_frac=0.15)
plt.figure(figsize=(10, 5))
plt.imshow(h3.T, origin='lower', cmap='terrain_r')
plt.colorbar(label=fr'Całkowita głębokość $h$ [m]')
plt.title('Profil początkowy – cieśnina z dwiema podwodnymi górami, fala płaska')
plt.xlabel('x [km]')
plt.ylabel('y [km]')
plt.tight_layout()
plt.savefig('3_1_Profil_poczatkowy_ciesnina_z_dwiema_podwodnymi_gorami_fala_plaska.pdf', format='pdf', bbox_inches='tight')
# show_anim(plot, range(len(out3['h'])), gif_file='3_2_Ewolucja_powierzchni_ciesnina_z_dwiema_podwodnymi_gorami_fala_plaska.gif')
# show_anim(plot_velocity_field, range(len(out3['h'])), gif_file='3_3_Pole_predkosci_ciesnina_z_dwiema_podwodnymi_gorami_fala_plaska.gif')

<Figure size 1000x500 with 2 Axes>

## Analiza zbieżności względem rozdzielczości siatki i kroku czasowego

### Zbocze z podwodną górą, zaburzenie skośne – siatka 2x gęstsza, 2x dłużej

In [ ]:
x1_hi, y1_hi, X1_hi, Y1_hi, b1_hi, z1_hi, h1_hi, out1_hi = run_jedna_gora_tilted_wave(
    nx=120, ny=80,             # 2x (60x40)
    nt=340,                    # 2x 170
    dt_over_dxy=(0.05, 0.05),outfreq=2)
    
plt.figure(figsize=(8, 4))
plt.imshow(h1_hi.T, origin='lower', cmap='terrain_r')
plt.colorbar(label=fr'Całkowita głębokość $h$ [m]')
plt.title('Profil początkowy – zbocze z podwodną górą, zaburzenie skośne')
plt.xlabel('x [km]')
plt.ylabel('y [km]')
plt.tight_layout()
plt.savefig('1_4_AZ_Profil_poczatkowy_zbocze_z_podwodna_gora_zaburzenie_skosne.pdf', format='pdf', bbox_inches='tight')
# show_anim(plot, range(len(out1_hi['h'])), gif_file='1_5_AZ_Ewolucja_powierzchni_zbocze_z_podwodna_gora_zaburzenie_skosne.gif')
# show_anim(plot_velocity_field, range(len(out1_hi['h'])), gif_file ='1_6_AZ_Pole_predkosci_zbocze_z_podwodna_gora_zaburzenie_skosne.gif')

<Figure size 800x400 with 2 Axes>

### Zbocze z podwodną górą, fala płaska – siatka 2x gęstsza, 2x dłużej

In [ ]:
x2_hi, y2_hi, X2_hi, Y2_hi, b2_hi, z2_hi, h2_hi, out2_hi = run_jedna_gora_flat_wave(
    nx=120, ny=80,             # 2x (60x40)
    nt=340,                    # 2x 170
    dt_over_dxy=(0.05, 0.05),
    outfreq=2,amp=1.5,x_frac=0.15)

plt.figure(figsize=(8, 4))
plt.imshow(h2_hi.T, origin='lower', cmap='terrain_r')
plt.colorbar(label=fr'Całkowita głębokość $h$ [m]')
plt.title('Profil początkowy – zbocze z podwodną górą, fala płaska')
plt.xlabel('x [km]')
plt.ylabel('y [km]')
plt.tight_layout()
plt.savefig('2_4_AZ_Profil_poczatkowy_zbocze_z_podwodna_gora_fala_plaska.pdf', format='pdf', bbox_inches='tight')

# show_anim(plot, range(len(out2_hi['h'])), gif_file='2_5_AZ_Ewolucja_powierzchni_zbocze_z_podwodna_gora_fala_plaska.gif')
# show_anim(plot_velocity_field, range(len(out2_hi['h'])), gif_file='2_6_AZ_Pole_predkosci_zbocze_z_podwodna_gora_fala_plaska.gif')

<Figure size 800x400 with 2 Axes>

### Cieśnina z dwiema podwodnymi górami, fala płaska – siatka 2x gęstsza, 2x dłużej

In [ ]:
x3_hi, y3_hi, X3_hi, Y3_hi, b3_hi, z3_hi, h3_hi, out3_hi = run_ciesnina(
    nx=120, ny=80,             # 2x (60x40)
    nt=340,                    # 2x 170
    dt_over_dxy=(0.05, 0.05),outfreq=2,amp=1.5,x_frac=0.15)
plt.figure(figsize=(10, 5))
plt.imshow(h3_hi.T, origin='lower', cmap='terrain_r')
plt.colorbar(label=fr'Całkowita głębokość $h$ [m]')
plt.title('Profil początkowy – cieśnina z dwiema podwodnymi górami, fala płaska')
plt.xlabel('x [km]')
plt.ylabel('y [km]')
plt.tight_layout()
plt.savefig('3_4_AZ_Profil_poczatkowy_ciesnina_z_dwiema_podwodnymi_gorami_fala_plaska.pdf', format='pdf', bbox_inches='tight')

# show_anim(plot, range(len(out3_hi['h'])), gif_file = '3_5_AZ_Ewolucja_powierzchni_ciesnina_z_dwiema_podwodnymi_gorami_fala_plaska.gif')
# show_anim(plot_velocity_field, range(len(out3_hi['h'])), gif_file = '3_6_AZ_Pole_predkosci_ciesnina_z_dwiema_podwodnymi_gorami_fala_plaska.gif')

<Figure size 1000x500 with 2 Axes>

In [16]:
def plot_3d_subplot(ax, out_data, bath_data, title, frame_idx=0):
    psi = out_data['h'][frame_idx] - bath_data
    xi, yi = np.indices(psi.shape)
    # Surface/Wireframe
    ax.plot_wireframe(xi+0.5, yi+0.5, psi, color='blue', linewidth=0.5, rstride=3, cstride=3)
    # Batymetria na dnie
    ax.contourf(xi+0.5, yi+0.5, bath_data, zdir='z', offset=-1.5, cmap='terrain_r', alpha=0.4)
    ax.set_zlim(-1.5, 1.5)
    ax.set_title(title, fontsize=10)
    ax.view_init(elev=45, azim=-60)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([]) 
    
def plot_vel_subplot(ax, out_data, title, frame_idx=0):
    h = out_data['h'][frame_idx]
    u = out_data['uh'][frame_idx] / (h + 1e-8)
    v = out_data['vh'][frame_idx] / (h + 1e-8)
    speed = np.sqrt(u**2 + v**2)
    
    im = ax.imshow(speed.T, origin='lower', cmap='viridis', aspect='equal')
    step = 2
    Y, X = np.mgrid[0:h.shape[1]:step, 0:h.shape[0]:step]
    ax.quiver(X, Y, u[::step, ::step].T, v[::step, ::step].T, color='white', scale=10, width=0.006)
    ax.set_title(title)
    return im

In [19]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), constrained_layout=True)
# Scenariusz 1: Zaburzenie skośne
im1 = axes[0].imshow(h1.T, origin='lower', cmap='terrain_r')
axes[0].set_title('(A) Zbocze + Góra (Fala skośna)'); axes[0].set_xlabel('x [km]'); axes[0].set_ylabel('y [km]')
plt.colorbar(im1, ax=axes[0], label='h [m]', fraction=0.046, pad=0.04)
# Scenariusz 2: Fala płaska
im2 = axes[1].imshow(h2.T, origin='lower', cmap='terrain_r')
axes[1].set_title('(B) Zbocze + Góra (Fala płaska)'); axes[1].set_xlabel('x [km]'); axes[1].set_yticks([])
plt.colorbar(im2, ax=axes[1], label='h [m]', fraction=0.046, pad=0.04)
# Scenariusz 3: Cieśnina
im3 = axes[2].imshow(h3.T, origin='lower', cmap='terrain_r')
axes[2].set_title('(C) Cieśnina (Dwie góry)'); axes[2].set_xlabel('x [km]'); axes[2].set_yticks([])
plt.colorbar(im3, ax=axes[2], label='h [m]', fraction=0.046, pad=0.04)
# plt.suptitle(fr"Rys. 1: Warunki początkowe - całkowita głębokość kolumny cieczy ($h = \zeta + b$)", fontsize=12);
plt.savefig('Rys_1_Warunki_poczatkowe_calkowita_glebokosc_kolumny_cieczy_h_zeta_plus_b.pdf', format='pdf', bbox_inches='tight');
plt.close(fig)

frame_idx = 80 
fig = plt.figure(figsize=(15, 4.5), constrained_layout=True)

ax1 = fig.add_subplot(131, projection='3d')
plot_3d_subplot(ax1, out1, b1, "(A) Fala skośna nad górą", frame_idx=frame_idx)
ax1.set_xlabel("x [km]")
ax1.set_ylabel("y [km]")
ax1.set_zlabel(r"$\zeta$ [m]")

ax2 = fig.add_subplot(132, projection='3d')
plot_3d_subplot(ax2, out2, b2, "(B) Fala płaska nad górą", frame_idx=frame_idx)
ax2.set_xlabel("x [km]")
ax2.set_ylabel("y [km]")
ax2.set_zlabel(r"$\zeta$ [m]")

ax3 = fig.add_subplot(133, projection='3d')
plot_3d_subplot(ax3, out3, b3, "(C) Fala w cieśninie", frame_idx=frame_idx)
ax3.set_xlabel("x [km]")
ax3.set_ylabel("y [km]")
ax3.set_zlabel(r"$\zeta$ [m]")
# plt.suptitle(fr"Rys. 2: Deformacja powierzchni swobodnej $\zeta$ w kroku {frame_idx}", fontsize=12)
plt.savefig('Rys_2_Deformacja_powierzchni_swobodnej_zeta_w_kroku_80.pdf',
            format='pdf', bbox_inches='tight');
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), constrained_layout=True)
im1 = plot_vel_subplot(axes[0], out1, "(A) Fala skośna", frame_idx=80)
im2 = plot_vel_subplot(axes[1], out2, "(B) Fala płaska", frame_idx=80)
im3 = plot_vel_subplot(axes[2], out3, "(C) Cieśnina", frame_idx=80)
cbar = fig.colorbar(im3, ax=axes, shrink=0.6, label='|u| [m/s]')
# plt.suptitle(f"Rys. 3: Pole prędkości cieczy w kroku {frame_idx}", fontsize=12); 
plt.savefig('Rys_3_Pole_predkosci_cieczy_w_kroku_80.pdf', format='pdf', bbox_inches='tight')
plt.close(fig)

C:\Users\Marcin\AppData\Local\Temp\ipykernel_31552\2647409115.py:39: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  plt.savefig('Rys_2_Deformacja_powierzchni_swobodnej_zeta_w_kroku_80.pdf',


\newpage

```{=latex}
\section*{Symulacja fal na morzu metanowym Tytana (księżyc Saturna): Wpływ ukształtowania dna na propagację fal}

#### 1. Wstęp i opis fizyczny
Celem projektu jest analiza wpływu zróżnicowanej batymetrii na dynamikę fal powierzchniowych na Tytanie. Symulacja opiera się na równaniach płytkiej wody (Shallow Water Equations - SWE), rozwiązywanych numerycznie przy użyciu solvera opartego na schemacie PyMPDATA.

**Pytanie badawcze:** 

**`Jak obecność podwodnych przeszkód (pojedyncza góra vs. cieśnina z dwiema górami) oraz charakter fali nadchodzącej (fala płaska vs. zaburzenie skośne) wpływają na amplitudę fali powierzchniowej oraz strukturę pola prędkości nad przeszkodami w warunkach grawitacji Tytana?`**

#### 1.2 Model fizyczny:
Płyn modelowany jest jako nieściśliwy, o stałej gęstości (ciekły metan), używając klasycznych równań płytkiej wody. Uwględniono następujące parametry: 

**Przyspieszenie grawitacyjne Tytana:** $g = 1.352 \text{ m/s}^2$.

**Batymetrię:** $b(x,y)$ mierzona dodatnio w dół.

**Wysokość swobodnej powierzchni:**  $\zeta (x,y,t)$ względem geoidy.

**Całkowitą głębokość kolumny cieczy:** $h=\zeta+b$. 

#### 1.3 Konfiguracja dna i warunki początkowe

W symulacjach wykorzystano model dna opadającego liniowo, na który nałożono przeszkody w postaci funkcji Gaussa. Zdefiniowano dwa układy batymetrii:

- **Zbocze z pojedynczą górą** – głębokość rośnie liniowo od lewej do prawej krawędzi domeny. W centrum znajduje się pojedyncze, podwodne wzniesienie (góra), którego szczyt znajduje się tuż pod powierzchnią wody.

- **Cieśnina (układ dwóch gór)** – na tym samym zboczu umieszczono dwa symetryczne wzniesienia po bokach osi przepływu. Tworzą one przesmyk (cieśninę) w centralnej części domeny.

*W obu przypadkach przeszkody są całkowicie zanurzone i nie tworzą wystających wysp.*


Ruch cieczy inicjowany jest przez dodanie do poziomu wody początkowego zaburzenia $\zeta(x,y)$. Rozpatrywane są dwa warianty:

- **fala płaska** – zaburzenie typu Gaussa wzdłuż osi $x$, praktycznie stałe w kierunku $y$,
- **fala skośna** – dwuwymiarowy Gauss z lekkim nachyleniem amplitudy w kierunku głębszej części.

W obu przypadkach początkowe prędkości przyjmuje się równe zeru, a fala zadawana jest w postaci zmiany $\zeta(x,y)$, co przekłada się na zmianę całkowitej głębokości $h = b + \zeta$.

#### 2 Metoda numeryczna

Do dyskretyzacji równań zastosowano bibliotekę `PyMPDATA`. Wykorzystano:

* **2D siatkę prostokątną** o rozmiarach:

  * bazowo: ($60 \times 40$) komórek,
  * dla analizy zbieżności: ($120 \times 80$) (siatka $2(\times)$ gęstsza).

* **Czas integracji:** 
    * bazowo: $n_t = 170$
    * dla analizy zbieżności: $n_t = 340$ (czas symulacji $2(\times)$ dłuższy)
* **Stały krok czasowy**, określony przez parametr $dt\_over\_dxy = (0.05, 0.05)$ tak, aby spełniony był warunek Couranta:
 $$
  |C| = \left| \frac{u,\Delta t}{\Delta x} \right| \le 1 $$, 
* **Warunki brzegowe typu**
  `boundary_conditions.Constant(value=0)`
  dla wszystkich *advectees*, co odpowiada „sztywnym ścianom” domeny.

* **Numeryczne źródło grawitacyjne** jest dodawane w funkcji `_apply_half_rhs`,
  a pola prędkości są aktualizowane poprzez podział pędu przez wysokość kolumny (h).
---

#### 3 Wizualizacja wyników

#### 3.1. Warunki początkowe
Na rys. \ref{fig:rys1_h} przedstawiono pola początkowej całkowitej głębokości $(h = \zeta + b)$ dla trzech scenariuszy bazowych na siatce ($60\times 40$):
* (A) zbocze z pojedynczą podwodną górą, zaburzenie skośne,
* (B) zbocze z pojedynczą podwodną górą, fala płaska,
* (C) cieśnina z dwiema podwodnymi górami, fala płaska.

```{=latex}
\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{Rys_1_Warunki_poczatkowe_calkowita_glebokosc_kolumny_cieczy_h_zeta_plus_b.pdf}
\caption{Warunki początkowe – całkowita głębokość kolumny cieczy $h=\zeta+b$ 
dla trzech konfiguracji batymetrii i zaburzeń początkowych.}
\label{fig:rys1_h}
\end{figure}

#### 3.2. Snapshoty powierzchni dla wybranej chwili
Na rys. \ref{fig:rys2_zeta} pokazano snapshoty powierzchni cieczy ($\zeta$) względem dna dla wybranej chwili symulacji. W trzech panelach przedstawiono odpowiednio:
* (A) falę skośną nad pojedynczą górą,
* (B) falę płaską nad tą samą batymetrią,
* (C) falę przechodzącą przez cieśninę z dwiema podwodnymi górami.

```{=latex}
\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{Rys_2_Deformacja_powierzchni_swobodnej_zeta_w_kroku_80.pdf}
\caption{Deformacja powierzchni swobodnej $\zeta$ w kroku czasowym $t_{80}$ 
dla trzech scenariuszy.}
\label{fig:rys2_zeta}
\end{figure}

#### 3.3. Snapshoty pola prędkości
Na rys. \ref{fig:rys3_vel} przedstawiono pola prędkości – moduł oraz wektory – dla tych samych chwil czasowych, co w rys. 2. Obserwuje się m.in.:
przyspieszenie przepływu nad szczytami podwodnych gór, silne ścinanie i zbieganie się strumieni w rejonie cieśniny oraz różnicę między bardziej zorganizowanym przepływem dla fali płaskiej a „poszarpanym” polem prędkości dla zaburzenia skośnego.


```{=latex}
\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{Rys_3_Pole_predkosci_cieczy_w_kroku_80.pdf}
\caption{Moduł prędkości $|\mathbf{u}|$ i wektory przepływu w kroku $t_{80}$.}
\label{fig:rys3_vel}
\end{figure}

#### 4. Analiza zbieżności
W celu oceny wpływu rozdzielczości czasowej i przestrzennej na wynik, wszystkie trzy scenariusze są powtarzane na siatce $(120\times 80)$ z dwukrotnie większą liczbą kroków czasowych $(n_t = 340)$, przy zachowaniu tego samego stosunku $\Delta t/\Delta x$.

Profil początkowy $(h)$ dla dwukrotnie większej siatki jest bardziej gładki, ale główne cechy geometryczne (zbocze, szczyty gór, cieśnina) pozostają niezmienione jakościowo.

Porównanie profili powierzchni i pól prędkości dla siatki bazowej oraz powiększonej prowadzi do następujących wniosków:

* ogólny kształt fali, położenie maksimów i minimów oraz struktura przepływu są bardzo zbliżone między siatką bazową a powiększoną,
* zwiększenie rozdzielczości powoduje lepsze uwidocznienie drobnych struktur i redukcję rozmycia numerycznego,
* nie obserwuje się istotnych zmian jakościowych obrazu przy przejściu z siatki $(60\times 40)$ na $(120\times 80)$.

#### Wnioski


Przeprowadzona symulacja jest stabilna numerycznie i nie obserwuje się żadnych znaczących artefaktów przy zmianie parametrów domeny obliczeniowej. Wszystkie podstawowe cechy $\zeta(x,y,t)$ i $\vec{u}(x,y,t)$ pozostają stabilne przy podwojeniu rozdzielczości przestrzenno‑czasowej. 
Zmodyfikowany solver równań płytkiej wody pozwala jakościowo ocenić wpływ batymetrii dna na propagację fal metanowych na Tytanie. W szczególności, przejście fal przez cieśninę z dwiema podwodnymi górami prowadzi do lokalnego skupienia przepływu i może sprzyjać zwiększaniu maksymalnych amplitud przy brzegu w porównaniu z przypadkiem pojedynczej przeszkody.

<!-- VERSION_TAG_2025_12_02_XYZ -->